In [1]:
from dbfread import DBF
import pandas as pd
import numpy as np
import plotly.express as px
import geopandas as gpd
import matplotlib as plt
import folium

from deep_translator import GoogleTranslator
import requests

In [2]:
# Reading and cleaning empty map data (no word freq, just languages and geometric data)

# Read map data, you will need this zip file to read the map data initially
empty_map = gpd.read_file("soc_071_world_languages.zip") 

# Cleaning map data
empty_map = empty_map.loc[:, ['COUNTRY', 'FIRST_OFFI']]
empty_map = empty_map.rename(columns={'COUNTRY': 'Country', 'FIRST_OFFI': 'Primary Language (based on 2015)'})
empty_map['Primary Language (based on 2015)'] = empty_map['Primary Language (based on 2015)'].replace({'Standard Chinese or Mandarin': 'Chinese'})

# Save as csv
empty_map.to_csv('empty_map.csv', index=False)

In [3]:
# Copy pasted from retrieval_function.py

languages = ['en', 'es', 'fr', 'de', 'it', 'ru', 'zh-CN', 'iw'] # list of languages used
NGRAM_API_URL = "https://books.google.com/ngrams/json" # API endpoint

# sets parameters for API call
def set_params(word, corpus):
    params = {'content': word,
              'year_start': 1500,
              'year_end': 2022,
              'corpus': corpus,
              'smoothing': 0,
              'case_insensitive': 'on'}
    return params

# function to get direct translations for words
def get_languages(input, input_lang):
    df = pd.DataFrame({'word' : input,
                       'language' : input_lang}, index = [0])
    for lang in languages:
        if lang != input_lang:
            translator = GoogleTranslator(source = input_lang, target = lang)
            new_entry = pd.DataFrame({'word' : translator.translate(input),
                            'language' : lang}, index = [0])
            df = pd.concat([df, new_entry], ignore_index = True)
            df['language'] = df['language'].replace('zh-CN', 'zh')
    return df

# gets frequency data from google ngram API
def get_frequency(df):
    years = list(range(1500, 2023))
    data = pd.DataFrame()
    for i in range(len(df)):
        response = requests.get(NGRAM_API_URL, params = set_params(df['word'][i], df['language'][i],), timeout = 30)
        if response.status_code == 200:
            x = response.json()
            if x:
                freq = x[0]['timeseries']
            else:
                freq = [0] * len(years)
        else:
            freq = [0] * len(years)
        data = pd.concat([data, pd.DataFrame({'word': df['word'][i], 
                                              'language' : df['language'][i],
                                              'year' : years, 
                                              'frequency' : freq})], ignore_index = True)
    return data

# main function to run (combines above functions)
def get_df(word, input_lang):
    return get_frequency(get_languages(word, input_lang))

In [4]:
# COPY STARTING HERE FOR STREAMLIT IMPLEMENTATION, MAKE SURE CSV IS IN FOLDER
empty_map = pd.read_csv('empty_map.csv')

In [5]:
# Slider determines year, I'm going to assume this to be a variable "year"
year = 2020

In [6]:
# User input is variable "input" and user selected input language is variable "input_lang"
input = "example"
input_lang = "en"

# This is the function call to get the frequency data
freq = get_df(input, input_lang) 

In [ ]:
# Adding freq and translations to map data
 
# Make a copy of empty_map called "map" and add freq data to this copy each time for map generation
map = empty_map.copy(deep=True)

# Filter frequency data for the selected year
freq_year = freq[freq['year'] == year]

# Combine frequency data and ngram translation into world map data
# Note this order is very important, as it is based on the retrieval functions being in this order
langs = [
    (map['Primary Language (based on 2015)'] == 'English'),
    (map['Primary Language (based on 2015)'] == 'Spanish'),
    (map['Primary Language (based on 2015)'] == 'French'),
    (map['Primary Language (based on 2015)'] == 'German'),
    (map['Primary Language (based on 2015)'] == 'Italian'),
    (map['Primary Language (based on 2015)'] == 'Russian'),
    (map['Primary Language (based on 2015)'] == 'Chinese'),
    (map['Primary Language (based on 2015)'] == 'Hebrew')
]
map['Frequency'] = np.select(langs, freq_year['frequency'], default=np.nan)
map['Ngram'] = np.select(langs, freq_year['word'], default='Unsupported language')

# Splitting unsupported languages into seperate dataframe (required for choropleth to show them)
map_na = map[map['Frequency'].isna()]
map_na['Frequency'].values[:] = 0
map = map[map['Frequency'].notna()]

In [8]:
# New map generation method using plotly instead of Geopandas

# No clicking popup, only hovering. I think it's ok to not have click popups, though if we really wanted to I believe there's a Streamlit/plotly extension for it, it might just be a bit complex
# With plotly, I can also no longer disable the wrapped world map, unless u disable panning horizontally altogether 
 
import plotly.graph_objects as go

# Tracing supported languages
fig = go.Figure(data=go.Choropleth(
    locationmode='country names',
    locations = map['Country'],
    z = map['Frequency'],
    colorscale = 'ice', # Can change color scale to other colors
    reversescale=True,
    autocolorscale=False,
    marker_line_color='white',
    marker_line_width=0.5,
    colorbar=dict(
        title='Relative Frequency',
        title_side='top',
        xanchor='center',
        xref='paper',
        x=0.85,
    ),
    customdata=list(zip(*[map['Ngram'], map['Primary Language (based on 2015)']])),
    hovertemplate='<b>Country:</b> %{location}<br>' 
        '<b>Primary Language (2015):</b> %{customdata[1]}<br>' 
        '<b>Ngram:</b> %{customdata[0]}<br>' 
        '<b>Relative Frequency:</b> %{z}<extra></extra>'
))

# Tracing unsupported languages/countries
fig.add_traces(go.Choropleth(
    locationmode='country names',
    locations = map_na['Country'],
    z = map_na['Frequency'],
    showscale=False,
    colorscale = [[0, 'darkgrey'], [1, 'darkgrey']],
    marker_line_color='white',
    marker_line_width=0.5,
    customdata=list(zip(*[map_na['Ngram'], map_na['Primary Language (based on 2015)']])),
    hovertemplate='<b>Country:</b> %{location}<br>' 
        '<b>Primary Language (2015):</b> %{customdata[1]}<br>' 
        '<b>Ngram:</b> %{customdata[0]}<br>' 
        '<b>Relative Frequency:</b> Unsupported Language<extra></extra>'
))

# Please play with the colors
fig.update_layout(
    title={
        'text': 'World Map Ngram Frequency', # Title
        'x': 0.5,
        'xanchor': 'center',
        'y': 0.82,
        'xref': 'paper',
    },
    geo=dict(
        scope='world',
        showframe=False,
        showcoastlines=False,
        showlakes=False,
        projection_type='natural earth',
        bgcolor='black', # Map background color/ocean color
        showland=True,
        landcolor="darkgray" # Land base color before applying freq (aka non-relevant language countries)
    ),
    annotations = [dict(
        x=0.5,
        y=-0.1,
        xref='paper',
        yref='paper',
        text='Source: <a href="https://resourcewatch.org/data/explore/soc_071_world_languages?section=Discover&selectedCollection=&zoom=1.8265083751374334&lat=-20.61119742910511&lng=10.069445314449105&pitch=0&bearing=0&basemap=dark&labels=light&layers=%255B%257B%2522dataset%2522%253A%252220662342-dcdd-4a42-9f58-bcc80217de71%2522%252C%2522opacity%2522%253A1%252C%2522layer%2522%253A%2522f2d76e6b-060d-4dc9-83ea-284bef6b2aae%2522%257D%255D&aoi=&page=1&sort=most-viewed&sortDirection=-1"> CIA World Factbook (2015)</a>',
        showarrow = False,
        #font=dict(color="red")
    )],      
    font=dict(color='white'), # Text color
    paper_bgcolor='black', # Outside map background color
)

fig.show()